# FTEC5660：Tree of Thoughts 可复现性作业

依次运行。使用 **CPU** 即可，不需要 GPU。默认 3 道题 × 3 次重复 × 两个配置，共 18 个任务运行；每个任务会调用模型多次。

作者已有输出的离线复核与自己的新 DeepSeek 调用分开记录。完整作业还需要自己的 GitHub 链接和真实 Zoom 录屏。


## 1. 上传并解压项目 ZIP

运行后选择下载到电脑的 `FTEC5660-Reproducibility-colab.zip`。


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, json, os

if sys.version_info < (3, 11):
    raise RuntimeError("请使用 Python 3.11 或更新的 Colab runtime。")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("请只上传一个项目 ZIP。")
with zipfile.ZipFile(zip_names[0]) as z:
    for info in z.infolist():
        target = (Path("/content") / info.filename).resolve()
        if not target.is_relative_to(Path("/content")):
            raise ValueError("ZIP 路径超出 /content，停止解压。")
    z.extractall("/content")
ROOT = Path("/content/FTEC5660-Reproducibility")
assert (ROOT / "reproduce.py").exists(), "没有找到项目根目录，请上传正确的 ZIP。"
RUN_DIR = ROOT / "experiment_runs/main"
print("项目目录：", ROOT)


## 2. 安装固定版本的依赖

使用 virtualenv 自带的 pip 创建或修复独立环境，兼容缺少 ensurepip 的 Colab runtime。安装工具放在临时目录，不必重启。


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/content/FTEC5660-Reproducibility")
ENV_DIR = Path("/content/ftec5660-repro-env")
TOOLS_DIR = Path("/content/ftec5660-env-tools")

# Use virtualenv's bundled pip instead of the runtime's ensurepip.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "--target", str(TOOLS_DIR), "virtualenv==20.35.4"],
    check=True,
)
builder_env = os.environ.copy()
builder_env["PYTHONPATH"] = str(TOOLS_DIR)
subprocess.run(
    [sys.executable, "-m", "virtualenv", "--no-periodic-update",
     "--python", sys.executable, str(ENV_DIR)],
    env=builder_env, check=True,
)

PY = str(ENV_DIR / "bin/python")
subprocess.run(
    [PY, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    cwd=ROOT, check=True,
)
print("独立环境已准备好：", PY)



## 3. 离线复核与源文件检查（不调用 API）

这一步应该重新算得作者已有记录的 **69/100**。这不是重新运行 GPT-4，也不是自己的新模型结果。


In [ ]:
subprocess.run(
    [PY, "reproduce.py", "--preflight", "--output", str(RUN_DIR)],
    cwd=ROOT, check=True,
)
display(json.loads((RUN_DIR / "archive_audit.json").read_text()))


## 4. 配置 Colab Secret

点击左侧钥匙图标，确认已有 `DEEPSEEK_API_KEY`，并打开当前笔记本的访问权限。**不要把 key 粘贴进代码或发到聊天里。**


In [ ]:
from google.colab import userdata
try:
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
except Exception as exc:
    raise RuntimeError("无法读取 Secret；检查名称 DEEPSEEK_API_KEY 和 Notebook access 开关。") from exc
config = json.loads((ROOT / "reproduction_config.json").read_text())
os.environ["DEEPSEEK_BASE_URL"] = config["base_url"]
os.environ["DEEPSEEK_MODEL"] = config["model"]
print("Secret 已配置；不会显示 key。模型：", config["model"])


## 5. API 连通性检查（会有一次小额真实调用）

只检查接口能否返回文字；这次检查不计入主实验成绩或 token 统计。


In [ ]:
probe_code = """
import sys
from pathlib import Path
sys.path.insert(0, 'src')
from tot import models
import os
models.reset_usage(Path(sys.argv[1]) / 'connectivity_check.jsonl')
reply = models.gpt('Reply with the single word READY.',
                   model=os.environ['DEEPSEEK_MODEL'],
                   temperature=0.7, max_tokens=12)
print('API reply:', reply[0])
"""
subprocess.run([PY, "-c", probe_code, str(RUN_DIR)], cwd=ROOT, check=True)
print("连通性检查通过。")


## 6. 运行真实实验

保持默认配置。程序逐题保存结果；若 Colab 中断，再运行本格会跳过已记录的任务。失败也保留，不挑选成功结果覆盖失败。

**此格会产生多次真实 API 调用。** 看到 `Running ...` 后等待即可。不要为了得到高分只重跑失败题。


In [ ]:
subprocess.run(
    [PY, "-u", "reproduce.py", "--output", str(RUN_DIR), "--resume"],
    cwd=ROOT, check=True,
)


## 7. 查看结果

把本格输出发给我，我可以继续检查结果并完善结论。错误信息和完整轨迹保存在 `traces/`；它们不包含 API key。


In [ ]:
import pandas as pd
display(pd.read_csv(RUN_DIR / "results.csv"))
display(pd.read_csv(RUN_DIR / "trial_summary.csv"))
summary = json.loads((RUN_DIR / "summary.json").read_text())
print(json.dumps(summary, indent=2))


## 8. 填信息，生成英文报告和录屏提纲

先填姓名、学号。GitHub 和录屏链接尚未生成时可以先空着，之后补上并重运行这一格。报告中的实测数字来自上面的文件，不是手填的。


In [ ]:
NAME = ""  # Your full name
STUDENT_ID = ""  # Your student ID
GITHUB_URL = ""  # Your own new reproducibility repository URL
PRESENTATION_URL = ""  # Actual Zoom recording link after recording

subprocess.run(
    [PY, "make_report.py", "--run-dir", str(RUN_DIR),
     "--name", NAME or "[Name to be completed]",
     "--student-id", STUDENT_ID or "[Student ID to be completed]",
     "--github-url", GITHUB_URL or "[GitHub URL to be completed]",
     "--presentation-url", PRESENTATION_URL or "[Zoom recording URL to be completed]"],
    cwd=ROOT, check=True,
)
report = RUN_DIR / ("report.pdf" if summary["complete"] else "report_draft.pdf")
print("报告：", report)
print("请从下一格下载的结果 ZIP 中打开 PDF。")
print((RUN_DIR / "submission_status.json").read_text())


## 9. 下载报告、原始结果和录屏提纲

ZIP 下载后保留。`report.pdf` 只有在主实验全部记录完后才生成。录屏提纲不能替代真实录屏。


In [ ]:
import shutil
artifact_zip = Path("/content/FTEC5660-Reproducibility-results")
shutil.make_archive(str(artifact_zip), "zip", root_dir=RUN_DIR)
files.download(str(artifact_zip) + ".zip")


## 10. 提交前

1. 检查报告姓名、学号、真实结果与结论，PDF 不超过 10 页。
2. 将代码和实测结果放到自己的新 GitHub 仓库，保留原作者归属和分次提交记录；不要覆盖 HW2 仓库。
3. 用 Zoom 录制最长 15 分钟的讲解，获取老师能访问的链接。
4. 按老师要求提交报告、GitHub 链接和录屏链接三项。

源项目已保留在 `README.upstream.md`，具体复现步骤见根目录 README；`assignment_zh.md` 是完整中文翻译。
